In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score, average_precision_score)
from imblearn.over_sampling import SMOTE        

df = pd.read_csv(r"C:\Users\SuNi\Downloads\creditcard.csv")
print(df["Class"].value_counts())

Class
0    284315
1       492
Name: count, dtype: int64


In [2]:
X = df.drop(columns="Class")
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_test = X_train.copy(), X_test.copy()

scaler = StandardScaler().fit(X_train[["Time", "Amount"]])
X_train[["Time", "Amount"]] = scaler.transform(X_train[["Time", "Amount"]])
X_test[["Time", "Amount"]] = scaler.transform(X_test[["Time", "Amount"]])

In [3]:
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)
print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE: ", y_res.value_counts().to_dict())

X_small = X_res.sample(n=50000, random_state=42)
y_small = y_res.loc[X_small.index]

Before SMOTE: {0: 227451, 1: 394}
After SMOTE:  {0: 227451, 1: 227451}


In [4]:
models = {
    "Logistic Regression": (LogisticRegression(max_iter=1000),
                            {"C": [0.01, 0.1, 1, 10]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=42),
                      {"max_depth": [3, 5, 8], "min_samples_leaf": [1, 10]}),
    "SVM (Linear)": (LinearSVC(max_iter=5000, random_state=42),
                     {"C": [0.01, 0.1, 1]}),
    "KNN": (KNeighborsClassifier(),
            {"n_neighbors": [3, 5, 7]}),
    "Random Forest": (RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
                      {"max_depth": [8, 12], "min_samples_leaf": [1, 10]}),
}

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rows = []
for name, (model, param_grid) in models.items():
    grid = GridSearchCV(model, param_grid, cv=skf,
                        scoring="average_precision", n_jobs=-1, verbose=1)
    grid.fit(X_small, y_small)
    print(name)
    print("  Best params:", grid.best_params_)
    print("  Best CV score:", round(grid.best_score_, 4))

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Logistic Regression
  Best params: {'C': 10}
  Best CV score: 0.9923
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Decision Tree
  Best params: {'max_depth': 8, 'min_samples_leaf': 10}
  Best CV score: 0.9924
Fitting 3 folds for each of 3 candidates, totalling 9 fits
SVM (Linear)
  Best params: {'C': 1}
  Best CV score: 0.9917
Fitting 3 folds for each of 3 candidates, totalling 9 fits
KNN
  Best params: {'n_neighbors': 7}
  Best CV score: 0.9966
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Random Forest
  Best params: {'max_depth': 12, 'min_samples_leaf': 1}
  Best CV score: 0.9998


In [5]:
best = grid.best_estimator_
pred = best.predict(X_test)
score = (best.predict_proba(X_test)[:, 1] if hasattr(best, "predict_proba")
             else best.decision_function(X_test))
rows.append({"model": name,
                 "Precision": precision_score(y_test, pred),
                 "Recall": recall_score(y_test, pred),
                 "F1": f1_score(y_test, pred),
                 "ROC-AUC": roc_auc_score(y_test, score),
                 "PR-AUC": average_precision_score(y_test, score)})

results = pd.DataFrame(rows).set_index("model")
print(results.round(4))

               Precision  Recall      F1  ROC-AUC  PR-AUC
model                                                    
Random Forest     0.4416  0.8878  0.5898   0.9801   0.759
